In [1]:
# CNN for posture classification
# dataset: office0_2
# classes: 
# -1: unknown or unlabeled; 
# 0: absence; 
# 1: presence, unclassified; 
# 2: standing; 
# 3: sitting by bed; 
# 4: sitting on bed; 
# 5: lying w/o cover; 
# 6: lying with cover

# We only keep data with label 0, 2, 3, 4, 5, 6
import sys
sys.path.append('/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/src/')

# import dataset class
from dataset.dataset import ThermalDataset

In [2]:
dataset = ThermalDataset('/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/data/office0_2')

# 1. check the distribution of labels
from collections import Counter
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

label_counts = Counter(dataset.annotations_expanded)
print("Label distribution: ", label_counts)

# 2. use only the data with label 0, 2, 3, 4, 5, 6 and remap them to 0..5
kept_labels = [0, 2, 3, 4, 5, 6]
label_to_index = {label: idx for idx, label in enumerate(kept_labels)}
index_to_label = {idx: label for label, idx in label_to_index.items()}
print("Label mapping:", label_to_index)

valid_indices = [
    i for i, label in enumerate(dataset.annotations_expanded)
    if label in kept_labels
]

# 3. create train / val / test splits with stratification
labels_valid = [dataset.annotations_expanded[i] for i in valid_indices]
print(len(labels_valid), len(valid_indices))
train_idx, test_idx = train_test_split(
    valid_indices,
    test_size=0.20,
    stratify=labels_valid,
    random_state=42,
)

labels_train = [dataset.annotations_expanded[i] for i in train_idx]
print(len(train_idx), len(labels_train))

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.20,
    stratify=labels_train,
    random_state=42,
)

print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}, Test size: {len(test_idx)}")

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

Label distribution:  Counter({5: 3813, 0: 3205, 4: 2612, 3: 1861, 6: 1806, 2: 1388, 1: 632, -1: 208})
Label mapping: {0: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5}
14685 14685
11748 11748
Train size: 9398, Val size: 2350, Test size: 2937


In [3]:
import torch
# Dataloader
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

In [4]:
# load one sample from the dataloader
for batch in train_dataloader:
    thermal = batch[0]['ira_highres']
    data = batch[1]
    print(thermal.shape, data.shape)
    break

torch.Size([32, 62, 80]) torch.Size([32])


In [5]:
import torch
import torch.nn as nn
import numpy as np

class SimpleIRA_CNN(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.3)
        
        # After conv + pooling: 128 * 7 * 10 = 8960
        self.fc1 = nn.Linear(128 * 7 * 10, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        # x comes in as (B, 60, 80)
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x)

        x = x.float()
        B, H, W = x.shape
        
        # Add channel dimension for Conv2d
        x = x.unsqueeze(1)                   # -> (B, 1, 60, 80)
        
        # Convolutional backbone
        x = self.relu(self.conv1(x))
        x = self.pool(x)                     # -> (B, 32, 30, 40)
        
        x = self.relu(self.conv2(x))
        x = self.pool(x)                     # -> (B, 64, 15, 20)
        
        x = self.relu(self.conv3(x))
        x = self.pool(x)                     # -> (B, 128, 7, 10)
        
        # Flatten
        x = x.view(B, -1)
        
        # Fully connected layers
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

In [6]:
# Train model
def remap_labels(labels, label_to_index, device):
    return torch.tensor(
        [label_to_index[int(label)] for label in labels],
        dtype=torch.long,
        device=device,
    )


def train_model(model, train_dataloader, val_dataloader, label_to_index, num_epochs=10, learning_rate=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch in train_dataloader:
            thermal = batch[0]['ira_highres'].to(device=device, dtype=torch.float32)
            labels = remap_labels(batch[1], label_to_index, device)
            
            outputs = model(thermal)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
        
        # Validate
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch in val_dataloader:
                thermal = batch[0]['ira_highres'].to(device=device, dtype=torch.float32)
                labels = remap_labels(batch[1], label_to_index, device)
                
                outputs = model(thermal)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        accuracy = correct / total
        print(f"Validation Accuracy: {accuracy:.4f}")


def test_model(model, test_dataloader, label_to_index):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_dataloader:
            thermal = batch[0]['ira_highres'].to(device=device, dtype=torch.float32)
            labels = remap_labels(batch[1], label_to_index, device)
            
            outputs = model(thermal)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")

model = SimpleIRA_CNN(num_classes=len(label_to_index))
# train_model(model, train_dataloader, val_dataloader, label_to_index, num_epochs=10, learning_rate=1e-3)
# test_model(model, test_dataloader, label_to_index)

In [7]:
# Export model
torch.save(model.state_dict(), '/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn.pth')

In [8]:
# load the model

model_loaded = SimpleIRA_CNN(num_classes=len(label_to_index))
model_loaded.load_state_dict(torch.load('/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/weights/posture_cnn.pth'))
model_loaded.eval() # set to evaluation mode

SimpleIRA_CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU(inplace=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=8960, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=6, bias=True)
)

In [35]:
def filter_dataset(dataset, label_to_index):
    valid_indices = [
        i for i, label in enumerate(dataset.annotations_expanded)
        if label in label_to_index
    ]
    
    filtered_dataset = torch.utils.data.Subset(dataset, valid_indices)
    
    return filtered_dataset



In [36]:
# load data from /Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/data/hall5
val_dataset = ThermalDataset('/Users/entomophile/Desktop/FYP/entry_exit_detection/presence_detection_workspace/data/hall5')
# keep only specified labels and remap labels in val_dataset to 0..5
val_dataset = filter_dataset(val_dataset, label_to_index)
# check if we still have label 1 in val_dataset
# print uniq labels in val_dataset
# val_labels = [val_dataset[i][1].item() for i in range(len(val_dataset))]
# print("Unique labels in val_dataset:", set(val_labels))


val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

test_model(model_loaded, val_dataloader, label_to_index)


Test Accuracy: 0.1621
